# 🕺 Jogo de Imitação de Poses Corporais — PDI (UFMA)

Este notebook executa o projeto **direto no navegador pelo Google Colab**, sem precisar instalar nada ou usar pendrive no computador da sala.

### Como usar na apresentação:
1. Execute a **Célula 1** para baixar o código do GitHub e preparar o ambiente na nuvem.
2. Execute a **Célula 2** e permita o acesso à webcam no navegador.
3. Ao finalizar, clique em **🛑 Encerrar & Salvar** na tela do jogo e execute a **Célula 3** para exibir os gráficos de desempenho.

In [ ]:
import os

if not os.path.exists("PDI-T5-Jogo_poses"):
    !git clone https://github.com/Eduardo-exe/PDI-T5-Jogo_poses.git

%cd PDI-T5-Jogo_poses
!git pull origin main -q
!pip install -q mediapipe opencv-python-headless numpy matplotlib
print("✅ Ambiente pronto na nuvem do Google Colab!")

In [ ]:
import cv2
import json
import time
import base64
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from IPython.display import display, HTML
from google.colab import output

from utils_pose import (
    extrair_vetor_angulos,
    calcular_similaridade,
    FiltroEstabilidade,
    CONEXOES_CORPO,
    ARTICULACOES,
    desenhar_esqueleto_alvo
)

MODELO = "pose_landmarker_full.task"
ARQUIVO_POSES = "poses_referencia.json"
LIMIAR_ACERTO = 65.0
TEMPO_CONFIRMACAO = 1.5
DURACAO_PREPARACAO = 5.0

with open(ARQUIVO_POSES, "r", encoding="utf-8") as f:
    poses = json.load(f)

base_options = python.BaseOptions(model_asset_path=MODELO)
options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.VIDEO
)
detector = vision.PoseLandmarker.create_from_options(options)

estado_jogo = {
    "indice_pose": 0,
    "pontuacao": 0,
    "inicio_acerto": None,
    "filtro": FiltroEstabilidade(tamanho_janela=5),
    "tempo_frame_anterior": time.perf_counter(),
    "estado": "PREPARACAO",
    "tempo_inicio_prep": time.perf_counter(),
    "tempo_inicio_pose_atual": time.perf_counter(),
    "tempo_inicio_jogo": time.perf_counter(),
    "inverter_h": True,
    "mostrar_metricas": True,
    "historico_fps": [],
    "historico_latencia": [],
    "historico_sim": [],
    "poses_concluidas_log": [],
    "dados_csv_linhas": [],
    "ultimo_ts_ms": 0,
    "encerrado": False,
}

def desenhar_painel_metricas(frame, fps, fps_medio, latencia_ms, lat_media, similaridade_suave, sim_bruta, angulos_usuario, landmarks_ok, acertos, t_corrido):
    h, w = frame.shape[:2]
    panel_w = min(295, int(w * 0.46))
    panel_h = 166
    px, py = 15, 15
    if px + panel_w > w or py + panel_h > h:
        return

    overlay = frame[py:py+panel_h, px:px+panel_w].copy()
    cv2.rectangle(overlay, (0, 0), (panel_w, panel_h), (18, 22, 30), -1)
    cv2.addWeighted(overlay, 0.76, frame[py:py+panel_h, px:px+panel_w], 0.24, 0, frame[py:py+panel_h, px:px+panel_w])
    cv2.rectangle(frame, (px, py), (px + panel_w, py + panel_h), (0, 200, 255), 1)

    cv2.putText(frame, "METRICAS PDI (TEMPO REAL)", (px + 8, py + 18), cv2.FONT_HERSHEY_SIMPLEX, 0.44, (0, 220, 255), 1)
    cv2.line(frame, (px + 6, py + 24), (px + panel_w - 6, py + 24), (60, 70, 90), 1)

    cor_fps = (0, 255, 100) if fps >= 25 else ((0, 200, 255) if fps >= 15 else (0, 80, 255))
    cor_lat = (0, 255, 100) if latencia_ms <= 60 else ((0, 200, 255) if latencia_ms <= 120 else (0, 80, 255))
    cor_sim = (0, 255, 100) if similaridade_suave >= LIMIAR_ACERTO else (0, 180, 255)

    erro_ang = max(0.0, (1.0 - similaridade_suave / 100.0) * 60.0) if landmarks_ok else 60.0
    lm_txt = "33/33 (OK)" if landmarks_ok else "0/33 (--)"
    cor_lm = (0, 255, 100) if landmarks_ok else (0, 100, 255)

    y_txt = py + 41
    dy = 17
    cv2.putText(frame, f"FPS: {fps:.1f}  (Med: {fps_medio:.1f})", (px + 8, y_txt), cv2.FONT_HERSHEY_SIMPLEX, 0.42, cor_fps, 1)
    y_txt += dy
    cv2.putText(frame, f"Latencia: {latencia_ms:.1f} ms  (Med: {lat_media:.1f})", (px + 8, y_txt), cv2.FONT_HERSHEY_SIMPLEX, 0.42, cor_lat, 1)
    y_txt += dy
    cv2.putText(frame, f"Similar.: {similaridade_suave:.1f}%  (Bruta: {sim_bruta:.1f}%)", (px + 8, y_txt), cv2.FONT_HERSHEY_SIMPLEX, 0.42, cor_sim, 1)
    y_txt += dy
    cv2.putText(frame, f"Erro Ang.: {erro_ang:.1f} deg  (Tol: 60)", (px + 8, y_txt), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (220, 220, 220), 1)
    y_txt += dy
    cv2.putText(frame, f"Landmarks: {lm_txt}  | Acertos: {acertos}", (px + 8, y_txt), cv2.FONT_HERSHEY_SIMPLEX, 0.40, cor_lm, 1)
    y_txt += dy

    if angulos_usuario is not None:
        oe, od = angulos_usuario.get("ombro_esq", 0.0), angulos_usuario.get("ombro_dir", 0.0)
        ce, cd = angulos_usuario.get("cotovelo_esq", 0.0), angulos_usuario.get("cotovelo_dir", 0.0)
        cv2.putText(frame, f"Ombros E/D: {oe:.0f} / {od:.0f} deg", (px + 8, y_txt), cv2.FONT_HERSHEY_SIMPLEX, 0.40, (200, 230, 255), 1)
        y_txt += dy
        cv2.putText(frame, f"Cotov. E/D: {ce:.0f} / {cd:.0f} deg  | {t_corrido:.0f}s", (px + 8, y_txt), cv2.FONT_HERSHEY_SIMPLEX, 0.40, (200, 230, 255), 1)
    else:
        cv2.putText(frame, "Ombros E/D: -- / -- deg", (px + 8, y_txt), cv2.FONT_HERSHEY_SIMPLEX, 0.40, (140, 140, 140), 1)
        y_txt += dy
        cv2.putText(frame, f"Cotov. E/D: -- / -- deg  | {t_corrido:.0f}s", (px + 8, y_txt), cv2.FONT_HERSHEY_SIMPLEX, 0.40, (140, 140, 140), 1)

def salvar_relatorios():
    st = estado_jogo
    tempo_total = time.perf_counter() - st["tempo_inicio_jogo"]
    fps_m = float(np.mean(st["historico_fps"])) if st["historico_fps"] else 0.0
    lat_m = float(np.mean(st["historico_latencia"])) if st["historico_latencia"] else 0.0
    sim_m = float(np.mean(st["historico_sim"])) if st["historico_sim"] else 0.0

    with open("metricas_partida.csv", "w", encoding="utf-8") as f:
        f.write("timestamp_ms,tempo_corrido_s,estado,pose_alvo,similaridade_pct,fps,latencia_ms,pontos\n")
        f.writelines(st["dados_csv_linhas"])

    rel = {
        "data_hora": time.strftime("%Y-%m-%d %H:%M:%S"),
        "duracao_total_segundos": round(tempo_total, 2),
        "pontuacao_final": st["pontuacao"],
        "total_poses_acertadas": len(st["poses_concluidas_log"]),
        "detalhes_poses": st["poses_concluidas_log"],
        "fps_medio": round(fps_m, 2),
        "latencia_media_ms": round(lat_m, 2),
        "similaridade_media_pct": round(sim_m, 2),
        "modo": "Google Colab WebRTC"
    }
    with open("relatorio_partida.json", "w", encoding="utf-8") as f:
        json.dump(rel, f, indent=2, ensure_ascii=False)

def processar_frame(img_b64, acao=""):
    st = estado_jogo
    if acao == "espelho":
        st["inverter_h"] = not st["inverter_h"]
    elif acao == "metricas":
        st["mostrar_metricas"] = not st["mostrar_metricas"]
    elif acao == "pular_prep" and st["estado"] == "PREPARACAO":
        st["estado"] = "JOGANDO"
        st["inicio_acerto"] = None
        st["filtro"].limpar()
        st["tempo_inicio_pose_atual"] = time.perf_counter()
    elif acao == "proxima":
        st["indice_pose"] = (st["indice_pose"] + 1) % len(poses)
        st["estado"] = "PREPARACAO"
        st["tempo_inicio_prep"] = time.perf_counter()
        st["tempo_inicio_pose_atual"] = time.perf_counter()
    elif acao == "encerrar":
        if not st["encerrado"]:
            st["encerrado"] = True
            salvar_relatorios()
        return {"encerrado": True, "pontos": st["pontuacao"], "acertos": len(st["poses_concluidas_log"])}

    t_inicio_ciclo = time.perf_counter()
    header, encoded = img_b64.split(",", 1)
    nparr = np.frombuffer(base64.b64decode(encoded), np.uint8)
    frame = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    if frame is None:
        return {"img": img_b64}

    if st["inverter_h"]:
        frame = cv2.flip(frame, 1)

    h, w, _ = frame.shape
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

    ts_ms = max(st["ultimo_ts_ms"] + 1, int(time.time() * 1000))
    st["ultimo_ts_ms"] = ts_ms

    resultado = detector.detect_for_video(mp_image, ts_ms)
    pose_atual = poses[st["indice_pose"]]
    similaridade_suave = 0.0
    sim_bruta = 0.0
    angulos_usuario = None
    landmarks_ok = False

    if resultado.pose_landmarks:
        landmarks_ok = True
        landmarks = resultado.pose_landmarks[0]
        angulos_usuario, _ = extrair_vetor_angulos(landmarks)
        sim_bruta = calcular_similaridade(angulos_usuario, pose_atual["angulos"])
        similaridade_suave = st["filtro"].atualizar(sim_bruta)

        cor = (0, 255, 0) if (st["estado"] == "JOGANDO" and similaridade_suave >= LIMIAR_ACERTO) else (0, 165, 255)
        for i, j in CONEXOES_CORPO:
            pt1 = (int(landmarks[i].x * w), int(landmarks[i].y * h))
            pt2 = (int(landmarks[j].x * w), int(landmarks[j].y * h))
            cv2.line(frame, pt1, pt2, cor, 3)
        for _, (a, b, c) in ARTICULACOES.items():
            for idx in (a, b, c):
                px, py = int(landmarks[idx].x * w), int(landmarks[idx].y * h)
                cv2.circle(frame, (px, py), 6, (0, 0, 255), -1)

        if st["estado"] == "JOGANDO":
            if similaridade_suave >= LIMIAR_ACERTO:
                if st["inicio_acerto"] is None:
                    st["inicio_acerto"] = time.perf_counter()
                if time.perf_counter() - st["inicio_acerto"] >= TEMPO_CONFIRMACAO:
                    st["pontuacao"] += 100
                    tempo_gasto = time.perf_counter() - st["tempo_inicio_pose_atual"]
                    st["poses_concluidas_log"].append({
                        "pose": pose_atual["nome"],
                        "tempo_segundos": round(tempo_gasto, 2),
                        "similaridade_final": round(similaridade_suave, 1),
                        "pontuacao_acumulada": st["pontuacao"]
                    })
                    st["indice_pose"] = (st["indice_pose"] + 1) % len(poses)
                    st["inicio_acerto"] = None
                    st["filtro"].limpar()
                    st["estado"] = "PREPARACAO"
                    st["tempo_inicio_prep"] = time.perf_counter()
                    st["tempo_inicio_pose_atual"] = time.perf_counter()
            else:
                st["inicio_acerto"] = None

    if st["estado"] == "PREPARACAO":
        tempo_passado = time.perf_counter() - st["tempo_inicio_prep"]
        segundos_restantes = max(0, int(DURACAO_PREPARACAO - tempo_passado) + 1)
        box_w = min(int(w * 0.85), 420)
        box_h = min(int(h * 0.75), 400)
        x0, y0 = (w - box_w) // 2, (h - box_h) // 2

        overlay = frame.copy()
        cv2.rectangle(overlay, (x0, y0), (x0 + box_w, y0 + box_h), (25, 25, 25), -1)
        cv2.addWeighted(overlay, 0.78, frame, 0.22, 0, frame)
        cv2.rectangle(frame, (x0, y0), (x0 + box_w, y0 + box_h), (0, 255, 255), 3)

        cv2.putText(frame, "PREPARE-SE!", (x0 + 20, y0 + 35), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
        cv2.putText(frame, f"Pose: {pose_atual['nome']}", (x0 + 20, y0 + 68), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2)
        cv2.putText(frame, f"Comeca em: {segundos_restantes}s", (x0 + 20, y0 + 100), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0, 255, 0), 2)
        desenhar_esqueleto_alvo(frame, pose_atual["pontos_2d"], x0 + 20, y0 + 112, box_w - 40, box_h - 140, cor=(0, 255, 255), espessura=3)

        if tempo_passado >= DURACAO_PREPARACAO:
            st["estado"] = "JOGANDO"
            st["inicio_acerto"] = None
            st["filtro"].limpar()
            st["tempo_inicio_pose_atual"] = time.perf_counter()
    else:
        box_w = max(110, min(140, int(w * 0.2)))
        box_h = int(box_w * 1.15)
        x0, y0 = w - box_w - 15, 15
        overlay_alvo = frame[y0:y0+box_h, x0:x0+box_w].copy()
        cv2.rectangle(overlay_alvo, (0, 0), (box_w, box_h), (25, 25, 25), -1)
        cv2.addWeighted(overlay_alvo, 0.75, frame[y0:y0+box_h, x0:x0+box_w], 0.25, 0, frame[y0:y0+box_h, x0:x0+box_w])
        cv2.rectangle(frame, (x0, y0), (x0 + box_w, y0 + box_h), (0, 255, 255), 2)
        cv2.putText(frame, "ALVO", (x0 + 8, y0 + 16), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (0, 255, 255), 1)
        desenhar_esqueleto_alvo(frame, pose_atual["pontos_2d"], x0 + 4, y0 + 18, box_w - 8, box_h - 22, cor=(0, 255, 255), espessura=2)

        hud_w, hud_h = min(280, int(w * 0.46)), 75
        hud_x, hud_y = 15, max(10, h - 20 - hud_h)
        overlay_hud = frame[hud_y:hud_y+hud_h, hud_x:hud_x+hud_w].copy()
        cv2.rectangle(overlay_hud, (0, 0), (hud_w, hud_h), (20, 20, 20), -1)
        cv2.addWeighted(overlay_hud, 0.75, frame[hud_y:hud_y+hud_h, hud_x:hud_x+hud_w], 0.25, 0, frame[hud_y:hud_y+hud_h, hud_x:hud_x+hud_w])
        cv2.rectangle(frame, (hud_x, hud_y), (hud_x + hud_w, hud_y + hud_h), (70, 70, 70), 1)

        cv2.putText(frame, f"Pose: {pose_atual['nome']}", (hud_x + 8, hud_y + 18), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 255), 1)
        cv2.putText(frame, f"Pontos: {st['pontuacao']}", (hud_x + 8, hud_y + 36), cv2.FONT_HERSHEY_SIMPLEX, 0.48, (255, 255, 255), 1)
        bar_w = min(150, hud_w - 70)
        bar_x, bar_y = hud_x + 8, hud_y + 44
        largura_preenchida = int((similaridade_suave / 100.0) * bar_w)
        cor_barra = (0, 255, 0) if similaridade_suave >= LIMIAR_ACERTO else (0, 165, 255)
        cv2.rectangle(frame, (bar_x, bar_y), (bar_x + bar_w, bar_y + 10), (50, 50, 50), -1)
        cv2.rectangle(frame, (bar_x, bar_y), (bar_x + largura_preenchida, bar_y + 10), cor_barra, -1)
        cv2.putText(frame, f"{similaridade_suave:.0f}%", (bar_x + bar_w + 6, bar_y + 10), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (255, 255, 255), 1)

        if st["inicio_acerto"] is not None:
            t_rest = max(0.0, TEMPO_CONFIRMACAO - (time.perf_counter() - st["inicio_acerto"]))
            cv2.putText(frame, f"SEGURE: {t_rest:.1f}s", (hud_x + 8, hud_y + 68), cv2.FONT_HERSHEY_SIMPLEX, 0.52, (0, 255, 0), 2)
        else:
            cv2.putText(frame, f"Meta: {LIMIAR_ACERTO:.0f}%", (hud_x + 8, hud_y + 68), cv2.FONT_HERSHEY_SIMPLEX, 0.38, (160, 160, 160), 1)

    t_fim_ciclo = time.perf_counter()
    latencia_ms = (t_fim_ciclo - t_inicio_ciclo) * 1000
    fps = 1.0 / max(t_fim_ciclo - st["tempo_frame_anterior"], 1e-6)
    st["tempo_frame_anterior"] = t_fim_ciclo
    st["historico_fps"].append(fps)
    st["historico_latencia"].append(latencia_ms)
    if st["estado"] == "JOGANDO" and similaridade_suave > 0:
        st["historico_sim"].append(similaridade_suave)

    t_corrido = t_fim_ciclo - st["tempo_inicio_jogo"]
    st["dados_csv_linhas"].append(f"{ts_ms},{t_corrido:.2f},{st['estado']},{pose_atual['nome']},{similaridade_suave:.1f},{fps:.1f},{latencia_ms:.1f},{st['pontuacao']}\n")

    if st["mostrar_metricas"] and not (st["estado"] == "PREPARACAO" and w < 750):
        fps_med = float(np.mean(st["historico_fps"][-30:]))
        lat_med = float(np.mean(st["historico_latencia"][-30:]))
        desenhar_painel_metricas(frame, fps, fps_med, latencia_ms, lat_med, similaridade_suave, sim_bruta, angulos_usuario, landmarks_ok, len(st["poses_concluidas_log"]), t_corrido)

    _, buf = cv2.imencode('.jpg', frame, [cv2.IMWRITE_JPEG_QUALITY, 75])
    b64_out = "data:image/jpeg;base64," + base64.b64encode(buf).decode("utf-8")
    return {"img": b64_out, "encerrado": False}

output.register_callback('notebook.processar_frame', processar_frame)

html_code = '''
<div style="font-family: sans-serif; background: #0f172a; padding: 15px; border-radius: 12px; color: white; max-width: 900px; text-align: center;">
  <h3 style="margin-top:0; color:#38bdf8;">🕺 Jogo de Imitação de Poses — PDI (Ao Vivo no Colab)</h3>
  <div style="margin-bottom: 10px; display: flex; gap: 8px; justify-content: center; flex-wrap: wrap;">
    <button onclick="window._acao='espelho'" style="padding: 8px 14px; border-radius: 6px; border: none; background: #1e293b; color: #38bdf8; cursor: pointer; font-weight: bold;">🪞 Espelho</button>
    <button onclick="window._acao='metricas'" style="padding: 8px 14px; border-radius: 6px; border: none; background: #1e293b; color: #34d399; cursor: pointer; font-weight: bold;">📊 Métricas</button>
    <button onclick="window._acao='pular_prep'" style="padding: 8px 14px; border-radius: 6px; border: none; background: #1e293b; color: #facc15; cursor: pointer; font-weight: bold;">⏭️ Pular Preparação</button>
    <button onclick="window._acao='proxima'" style="padding: 8px 14px; border-radius: 6px; border: none; background: #1e293b; color: #e2e8f0; cursor: pointer; font-weight: bold;">➡️ Próxima Pose</button>
    <button onclick="window._acao='encerrar'" style="padding: 8px 14px; border-radius: 6px; border: none; background: #dc2626; color: white; cursor: pointer; font-weight: bold;">🛑 Encerrar & Salvar</button>
  </div>
  <img id="display_jogo" style="width: 100%; max-width: 850px; border-radius: 8px; border: 2px solid #334155; background: #000;" />
  <div id="status_jogo" style="margin-top: 8px; color: #94a3b8; font-size: 14px;">Solicitando acesso à webcam...</div>
</div>
<script>
(async function() {
  window._acao = "";
  const img = document.getElementById('display_jogo');
  const status = document.getElementById('status_jogo');
  const video = document.createElement('video');
  const canvas = document.createElement('canvas');
  let stream = null;

  try {
    stream = await navigator.mediaDevices.getUserMedia({ video: { width: 854, height: 480 } });
    video.srcObject = stream;
    await video.play();
    canvas.width = video.videoWidth || 640;
    canvas.height = video.videoHeight || 480;
    status.innerText = "🎥 Webcam conectada! Processando em tempo real...";
  } catch (err) {
    status.innerText = "❌ Erro ao acessar webcam: " + err.message;
    return;
  }

  const ctx = canvas.getContext('2d');
  let rodando = true;

  while (rodando) {
    ctx.drawImage(video, 0, 0, canvas.width, canvas.height);
    const dataUrl = canvas.toDataURL('image/jpeg', 0.72);
    const acaoAtual = window._acao;
    window._acao = "";

    try {
      const resp = await google.colab.kernel.invokeFunction('notebook.processar_frame', [dataUrl, acaoAtual], {});
      const data = resp.data['application/json'];
      if (data.encerrado) {
        rodando = false;
        stream.getTracks().forEach(t => t.stop());
        status.innerText = `✅ Partida encerrada! Pontos: ${data.pontos} | Acertos: ${data.acertos}. Execute a Célula 3 para ver os gráficos!`;
        break;
      }
      if (data.img) img.src = data.img;
    } catch (e) {
      status.innerText = "⚠️ Conexão interrompida: " + e;
      break;
    }
  }
})();
</script>
'''
display(HTML(html_code))

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

with open("relatorio_partida.json", "r", encoding="utf-8") as f:
    rel = json.load(f)

print("="*55)
print("📊 RELATÓRIO DA PARTIDA — PDI (UFMA)")
print("="*55)
print(f"🏆 Pontuação Final: {rel['pontuacao_final']} pontos")
print(f"🎯 Poses Acertadas: {rel['total_poses_acertadas']}")
print(f"⏱️  Duração Total:   {rel['duracao_total_segundos']} s")
print(f"⚡ FPS Médio:       {rel['fps_medio']} FPS")
print(f"⏱️  Latência Média:  {rel['latencia_media_ms']} ms")
print("="*55)

df = pd.read_csv("metricas_partida.csv")
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(df["tempo_corrido_s"], df["similaridade_pct"], color="#0284c7", label="Similaridade (%)")
axes[0].axhline(65, color="#16a34a", linestyle="--", label="Limiar (65%)")
axes[0].set_title("Similaridade Angular ao Longo do Tempo")
axes[0].set_xlabel("Tempo (s)")
axes[0].set_ylabel("Similaridade (%)")
axes[0].set_ylim(0, 105)
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(df["tempo_corrido_s"], df["latencia_ms"], color="#ea580c", label="Latência (ms)")
axes[1].set_title("Latência de Processamento por Quadro")
axes[1].set_xlabel("Tempo (s)")
axes[1].set_ylabel("Latência (ms)")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()